In [ ]:
# проверяем, что видеокарта доступна
import subprocess
from pathlib import Path

result = subprocess.run(["nvidia-smi"], text=True, capture_output=True)
print(result.stdout if result.returncode == 0 else result.stderr)
if result.returncode != 0:
    raise RuntimeError("Для воспроизведения решения нужен CUDA GPU.")

# считаем текущую папку корнем проекта
PROJECT_ROOT = Path.cwd().resolve()
print("Корневая папка проекта:", PROJECT_ROOT)


In [ ]:
# ставим все зависимости из одного файла
import subprocess
import sys

requirements_path = PROJECT_ROOT / "requirements.txt"
if not requirements_path.exists():
    raise FileNotFoundError(
        f"Не найден файл с зависимостями: {requirements_path}"
    )

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(requirements_path),
    ]
)


In [ ]:
# задаем пути и основные настройки запуска
from pathlib import Path
import shutil
import tempfile

PROJECT_DIR = PROJECT_ROOT
# здесь лежат исходные данные
DATA_DIR = PROJECT_DIR / "datasets"
ARTIFACTS_DIR = PROJECT_DIR / "dense_artifacts"
MODELS_DIR = PROJECT_DIR / "models"
LOCAL_DATA_DIR = Path(tempfile.gettempdir()) / "avito_finetune_data"
LOCAL_CHECKPOINT_DIR = Path(tempfile.gettempdir()) / "e5_small_avito_checkpoints"
LOCAL_FINAL_MODEL_DIR = Path(tempfile.gettempdir()) / "e5_small_avito_model"

BASE_MODEL_NAME = "intfloat/multilingual-e5-small"
FINETUNED_MODEL_DIR = MODELS_DIR / "multilingual_e5_small_avito_v1"
MODEL_COMPLETE_PATH = FINETUNED_MODEL_DIR / "training_complete.json"

# эти параметры управляют обучением и размером батчей
MAX_SEQ_LENGTH = 256
TRAIN_BATCH_SIZE = 64
ENCODING_BATCH_SIZE = 128
SEARCH_BATCH_SIZE = 64
LEARNING_RATE = 2e-5
NUM_TRAIN_EPOCHS = 1
WARMUP_RATIO = 0.1
MAX_POSITIVES_PER_QUERY = 3
VALIDATION_QUERY_COUNT = 2_452
RANDOM_SEED = 42
USE_FP16 = True

DENSE_GLOBAL_DEPTH = 100
DENSE_LOCAL_DEPTH = 50
BM25_GLOBAL_DEPTH = 100
BM25_LOCAL_DEPTH = 50

# собираем полные пути ко всем входным файлам
SOURCE_TRAIN_PATH = DATA_DIR / "train.parquet"
SOURCE_ITEMS_PATH = DATA_DIR / "benchmark_items.parquet"
SOURCE_QUERIES_PATH = DATA_DIR / "benchmark_queries.parquet"
LOCAL_TRAIN_PATH = LOCAL_DATA_DIR / "train.parquet"
LOCAL_ITEMS_PATH = LOCAL_DATA_DIR / "benchmark_items.parquet"
LOCAL_QUERIES_PATH = LOCAL_DATA_DIR / "benchmark_queries.parquet"

RUN_NAME = f"multilingual_e5_small_avito_v1_len{MAX_SEQ_LENGTH}"
EMBEDDINGS_PATH = ARTIFACTS_DIR / f"{RUN_NAME}_item_embeddings.npy"
PROGRESS_PATH = ARTIFACTS_DIR / f"{RUN_NAME}_progress.json"
METRICS_PATH = (
    ARTIFACTS_DIR
    / "finetuned_e5_small_validation_metrics.csv"
)

# сразу проверяем, что нужные файлы на месте
for path in (SOURCE_TRAIN_PATH, SOURCE_ITEMS_PATH, SOURCE_QUERIES_PATH):
    if not path.exists():
        raise FileNotFoundError(f"Не найден файл: {path}")

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# копируем parquet во временную папку, чтобы быстрее читать его во время запуска
for source, destination in (
    (SOURCE_TRAIN_PATH, LOCAL_TRAIN_PATH),
    (SOURCE_ITEMS_PATH, LOCAL_ITEMS_PATH),
    (SOURCE_QUERIES_PATH, LOCAL_QUERIES_PATH),
):
    if not destination.exists() or destination.stat().st_size != source.stat().st_size:
        shutil.copy2(source, destination)
        print(
            f"Скопирован {source.name}: "
            f"{destination.stat().st_size / 1024**2:.1f} МБ"
        )
    else:
        print(f"Уже скопирован: {destination}")

In [ ]:
# собираем validation и пары для обучения
import hashlib
import json
import random
import numpy as np
import pandas as pd
import torch

# фиксируем случайность, чтобы повторный запуск давал тот же результат
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

QUERY_COLUMNS = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]
ITEM_TEXT_COLUMNS = [
    "item_title_raw",
    "item_infm_params_text",
    "item_description_raw",
]

# приводим запросы к одному виду перед сравнением
def normalize_text(series: pd.Series) -> pd.Series:
    return (
        series.fillna("")
        .astype(str)
        .str.lower()
        .str.replace("ё", "е", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# убираем лишние пробелы из текстов объявлений
def clean_item_text(series: pd.Series) -> pd.Series:
    return (
        series.fillna("")
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# делаем стабильный короткий id из текста
def stable_hash(value: str) -> str:
    return hashlib.blake2b(
        value.encode("utf-8"),
        digest_size=8,
    ).hexdigest()

# собираем единый текст объявления для dense модели
def build_passages(frame: pd.DataFrame) -> pd.Series:
    titles = clean_item_text(frame["item_title_raw"])
    params = clean_item_text(frame["item_infm_params_text"]).str.slice(0, 700)
    descriptions = (
        clean_item_text(frame["item_description_raw"])
        .str.slice(0, 1_200)
    )
    return (
        "passage: Заголовок: "
        + titles
        + ". Параметры: "
        + params
        + ". Описание: "
        + descriptions
    )

# если модель уже сохранена, длинные тексты train читать не нужно
training_already_complete = MODEL_COMPLETE_PATH.exists()
train_columns = QUERY_COLUMNS + ["item_id"]
if not training_already_complete:
    train_columns += ITEM_TEXT_COLUMNS

# читаем только те колонки, которые нужны на этом шаге
train = pd.read_parquet(
    LOCAL_TRAIN_PATH,
    columns=train_columns,
)
items = pd.read_parquet(
    LOCAL_ITEMS_PATH,
    columns=[
        "item_id",
        "item_location_id",
        *ITEM_TEXT_COLUMNS,
    ],
)
benchmark_queries = pd.read_parquet(
    LOCAL_QUERIES_PATH,
    columns=["search_query"],
)

train["item_id"] = train["item_id"].astype(str)
items["item_id"] = items["item_id"].astype(str)
train["normalized_query"] = normalize_text(train["search_query"])
train["normalized_params"] = normalize_text(
    train["search_infm_params_text"]
)
benchmark_query_texts = set(
    normalize_text(benchmark_queries["search_query"])
)
corpus_item_ids = set(items["item_id"])

# создаем validation тем же способом, что и раньше
eligible = train[
    train["item_id"].isin(corpus_item_ids)
    & ~train["normalized_query"].isin(benchmark_query_texts)
].copy()
# соединяем признаки запроса в одну стабильную подпись
signature_text = (
    eligible["normalized_query"]
    + "\x1f" + eligible["search_location_id"].astype(str)
    + "\x1f" + eligible["search_is_delivery_search"].astype(str)
    + "\x1f" + eligible["normalized_params"]
    + "\x1f" + eligible["search_category"].astype(str)
)
eligible["validation_query_id"] = signature_text.map(stable_hash)
pairs = eligible.drop_duplicates(["validation_query_id", "item_id"])

query_variants = pairs.drop_duplicates("validation_query_id").copy()
query_variants["variant_order"] = (
    query_variants["validation_query_id"].map(stable_hash)
)
representatives = (
    query_variants
    .sort_values("variant_order", kind="stable")
    .drop_duplicates("normalized_query")
)
representatives["text_order"] = (
    representatives["normalized_query"].map(stable_hash)
)
validation_queries = (
    representatives
    .sort_values("text_order", kind="stable")
    .head(VALIDATION_QUERY_COUNT)
    .reset_index(drop=True)
)
selected_query_ids = set(validation_queries["validation_query_id"])
qrels = pairs[
    pairs["validation_query_id"].isin(selected_query_ids)
][["validation_query_id", "item_id"]].copy()

if len(validation_queries) != VALIDATION_QUERY_COUNT:
    raise ValueError("Не удалось воспроизвести validation-выборку")
if set(validation_queries["normalized_query"]) & benchmark_query_texts:
    raise ValueError("В validation попали тексты benchmark-запросов")

# убираем validation запросы из обучения, чтобы оценка была честной
validation_texts = set(validation_queries["normalized_query"])
training_pairs = train[
    ~train["normalized_query"].isin(validation_texts)
].drop_duplicates(["normalized_query", "item_id"]).copy()
training_pairs["pair_order"] = (
    training_pairs["normalized_query"]
    + "\x1f" + training_pairs["item_id"]
).map(stable_hash)
training_pairs = (
    training_pairs
    .sort_values("pair_order", kind="stable")
    .groupby("normalized_query", sort=False)
    .head(MAX_POSITIVES_PER_QUERY)
    .reset_index(drop=True)
)

if set(training_pairs["normalized_query"]) & validation_texts:
    raise ValueError("Обучение пересекается с validation по тексту запроса")
if (training_pairs["normalized_query"] == "").any():
    raise ValueError("В обучение попал пустой запрос")

if training_already_complete:
    training_anchors = None
    training_positives = None
    print("Готовая модель найдена: обучающие тексты повторно не создаются")
else:
    training_anchors = (
        "query: " + training_pairs["normalized_query"]
    ).tolist()
    training_positives = build_passages(training_pairs).tolist()

# сохраняем отпечаток обучающих пар для проверки артефактов
training_pairs_hash = hashlib.sha256(
    "\n".join(
        training_pairs["normalized_query"]
        + "\x1f" + training_pairs["item_id"]
    ).encode("utf-8")
).hexdigest()

item_ids = items["item_id"].tolist()
item_index_by_id = {
    item_id: index
    for index, item_id in enumerate(item_ids)
}
validation_query_ids = (
    validation_queries["validation_query_id"].tolist()
)
validation_locations = validation_queries[
    "search_location_id"
].to_numpy(dtype=np.int64)
relevant_indices = {
    query_id: {
        item_index_by_id[item_id]
        for item_id in group["item_id"]
    }
    for query_id, group in qrels.groupby(
        "validation_query_id",
        sort=False,
    )
}
validation_dense_texts = (
    "query: " + validation_queries["normalized_query"]
).tolist()

print(f"Validation-запросов: {len(validation_queries):,}")
print(f"Положительных validation-пар: {len(qrels):,}")
print(f"Обучающих пар: {len(training_pairs):,}")
print(
    "Уникальных обучающих запросов:",
    training_pairs["normalized_query"].nunique(),
)
print(
    "Уникальных выбранных объявлений:",
    training_pairs["item_id"].nunique(),
)
print(
    "Benchmark-текстов, встречающихся в обучении:",
    training_pairs["normalized_query"].isin(
        benchmark_query_texts
    ).groupby(training_pairs["normalized_query"]).any().sum(),
)

In [ ]:
# загружаем готовую модель или дообучаем новую
import gc
import sentence_transformers
from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)
try:
    from sentence_transformers.training_args import BatchSamplers
except ImportError:
    from sentence_transformers.sentence_transformer.training_args import (
        BatchSamplers,
    )

# запоминаем настройки, с которыми должна быть обучена модель
expected_training_metadata = {
    "base_model_name": BASE_MODEL_NAME,
    "max_seq_length": MAX_SEQ_LENGTH,
    "learning_rate": LEARNING_RATE,
    "num_train_epochs": NUM_TRAIN_EPOCHS,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "max_positives_per_query": MAX_POSITIVES_PER_QUERY,
    "training_pairs": len(training_pairs),
    "training_pairs_sha256": training_pairs_hash,
    "random_seed": RANDOM_SEED,
}

# готовую модель используем только после проверки ее настроек
if MODEL_COMPLETE_PATH.exists():
    training_metadata = json.loads(
        MODEL_COMPLETE_PATH.read_text(encoding="utf-8")
    )
    for key, expected_value in expected_training_metadata.items():
        if training_metadata.get(key) != expected_value:
            raise ValueError(
                f"Сохранённая модель обучена с другими настройками: {key}"
            )
    model = SentenceTransformer(
        str(FINETUNED_MODEL_DIR),
        device="cuda",
    )
    model.max_seq_length = MAX_SEQ_LENGTH
    print("Загружена готовая fine-tuned модель:", FINETUNED_MODEL_DIR)
else:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA недоступна")

    model = SentenceTransformer(
        BASE_MODEL_NAME,
        device="cuda",
    )
    model.max_seq_length = MAX_SEQ_LENGTH

    # превращаем пары запрос и объявление в датасет для trainer
    train_dataset = Dataset.from_dict(
        {
            "anchor": training_anchors,
            "positive": training_positives,
        }
    )
    train_loss = losses.MultipleNegativesRankingLoss(model)

    training_args = SentenceTransformerTrainingArguments(
        output_dir=str(LOCAL_CHECKPOINT_DIR),
        num_train_epochs=NUM_TRAIN_EPOCHS,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        warmup_ratio=WARMUP_RATIO,
        fp16=USE_FP16,
        bf16=False,
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        save_strategy="epoch",
        save_total_limit=1,
        logging_steps=50,
        report_to="none",
        seed=RANDOM_SEED,
        data_seed=RANDOM_SEED,
        dataloader_num_workers=2,
    )
    # запускаем одну эпоху обучения с отрицательными примерами из батча
    trainer = SentenceTransformerTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        loss=train_loss,
    )
    train_result = trainer.train()

    # сохраняем модель отдельно от временных checkpoint файлов
    if LOCAL_FINAL_MODEL_DIR.exists():
        shutil.rmtree(LOCAL_FINAL_MODEL_DIR)
    model.save_pretrained(str(LOCAL_FINAL_MODEL_DIR))
    shutil.copytree(
        LOCAL_FINAL_MODEL_DIR,
        FINETUNED_MODEL_DIR,
        dirs_exist_ok=True,
    )

    training_metadata = {
        **expected_training_metadata,
        "sentence_transformers_version": sentence_transformers.__version__,
        "training_metrics": {
            key: float(value)
            for key, value in train_result.metrics.items()
            if isinstance(value, (int, float))
        },
    }
    MODEL_COMPLETE_PATH.write_text(
        json.dumps(
            training_metadata,
            ensure_ascii=False,
            indent=2,
        ) + "\n",
        encoding="utf-8",
    )
    print("Fine-tuned модель сохранена:", FINETUNED_MODEL_DIR)

print("Sentence Transformers:", sentence_transformers.__version__)
print("Модель:", FINETUNED_MODEL_DIR)
print("Максимальная длина:", model.max_seq_length)

# удаляем тяжелые объекты обучения, чтобы освободить память
heavy_training_objects = [
    "train",
    "eligible",
    "pairs",
    "query_variants",
    "representatives",
    "training_pairs",
    "training_anchors",
    "training_positives",
    "benchmark_queries",
    "qrels",
    "train_dataset",
    "train_loss",
    "training_args",
    "trainer",
    "train_result",
]
for variable_name in heavy_training_objects:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()
torch.cuda.empty_cache()
print("Обучающие объекты удалены из RAM")

In [ ]:
# строим эмбеддинги всех объявлений и сохраняем прогресс
import gc
from tqdm.auto import tqdm

# заранее собираем тексты объявлений в том же формате, что был при обучении
benchmark_item_texts = build_passages(items).tolist()
item_ids_hash = hashlib.sha256(
    "\n".join(item_ids).encode("utf-8")
).hexdigest()

if USE_FP16:
    model.half()

embedding_dimension = model.get_sentence_embedding_dimension()
if embedding_dimension is None:
    raise ValueError("Не удалось определить размерность эмбеддингов")

# эти поля защищают от случайного смешивания разных запусков
expected_progress_metadata = {
    "model_name": "multilingual_e5_small_avito_v1",
    "base_model_name": BASE_MODEL_NAME,
    "max_seq_length": MAX_SEQ_LENGTH,
    "embedding_dimension": int(embedding_dimension),
    "item_count": len(item_ids),
    "item_ids_sha256": item_ids_hash,
    "training_pairs_sha256": training_pairs_hash,
}

def save_embedding_progress(completed_items: int) -> None:
    payload = {
        **expected_progress_metadata,
        "completed_items": completed_items,
    }
    temporary_path = PROGRESS_PATH.with_suffix(".tmp")
    temporary_path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    temporary_path.replace(PROGRESS_PATH)

# если расчет прервался, продолжаем с последнего сохраненного объявления
if EMBEDDINGS_PATH.exists() and PROGRESS_PATH.exists():
    progress = json.loads(
        PROGRESS_PATH.read_text(encoding="utf-8")
    )
    for key, expected_value in expected_progress_metadata.items():
        if progress.get(key) != expected_value:
            raise ValueError(
                f"Существующие эмбеддинги имеют другие настройки: {key}"
            )
    completed_items = int(progress.get("completed_items", 0))
    item_embeddings = np.lib.format.open_memmap(
        EMBEDDINGS_PATH,
        mode="r+",
    )
    expected_shape = (len(item_ids), embedding_dimension)
    if item_embeddings.shape != expected_shape:
        raise ValueError(
            f"Некорректная форма эмбеддингов: {item_embeddings.shape}"
        )
    print(f"Продолжаем с объявления {completed_items:,}")
elif EMBEDDINGS_PATH.exists() or PROGRESS_PATH.exists():
    raise RuntimeError(
        "Найден только один файл прогресса. "
        "Удалите оба fine-tuned артефакта эмбеддингов и начните заново."
    )
else:
    item_embeddings = np.lib.format.open_memmap(
        EMBEDDINGS_PATH,
        mode="w+",
        dtype=np.float32,
        shape=(len(item_ids), embedding_dimension),
    )
    completed_items = 0
    save_embedding_progress(completed_items)

# кодируем корпус небольшими батчами и после каждого сохраняем результат
for start in tqdm(
    range(
        completed_items,
        len(benchmark_item_texts),
        ENCODING_BATCH_SIZE,
    ),
    desc="Кодирование корпуса fine-tuned E5-small",
):
    end = min(
        start + ENCODING_BATCH_SIZE,
        len(benchmark_item_texts),
    )
    batch_embeddings = model.encode(
        benchmark_item_texts[start:end],
        batch_size=ENCODING_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
        device="cuda",
    ).astype(np.float32)
    item_embeddings[start:end] = batch_embeddings
    item_embeddings.flush()
    save_embedding_progress(end)

del benchmark_item_texts
gc.collect()

print("Эмбеддинги готовы:", item_embeddings.shape)
print("Файл:", EMBEDDINGS_PATH)

In [ ]:
# получаем глобальную и локальную dense выдачу
# кодируем validation запросы той же моделью
query_embeddings = model.encode(
    validation_dense_texts,
    batch_size=ENCODING_BATCH_SIZE,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
    device="cuda",
).astype(np.float32)

# переносим эмбеддинги на gpu для быстрого точного поиска
corpus_tensor = torch.from_numpy(
    np.asarray(item_embeddings)
).to("cuda")
query_tensor = torch.from_numpy(query_embeddings).to("cuda")
item_locations = items["item_location_id"].to_numpy(dtype=np.int64)
item_locations_tensor = torch.from_numpy(
    item_locations
).to("cuda")
dense_global_rankings = []
dense_local_rankings = []

# градиенты здесь не нужны, потому что модель уже обучена
with torch.inference_mode():
    for start in tqdm(
        range(
            0,
            len(validation_queries),
            SEARCH_BATCH_SIZE,
        ),
        desc="Fine-tuned dense rankings",
    ):
        end = min(
            start + SEARCH_BATCH_SIZE,
            len(validation_queries),
        )
        scores = query_tensor[start:end] @ corpus_tensor.T
        global_indices = torch.topk(
            scores,
            k=DENSE_GLOBAL_DEPTH,
            dim=1,
            largest=True,
            sorted=True,
        ).indices.cpu().numpy()

        # для локальной выдачи скрываем объявления из других локаций
        batch_locations = torch.from_numpy(
            validation_locations[start:end]
        ).to("cuda")
        same_location = batch_locations[:, None].eq(
            item_locations_tensor[None, :]
        )
        local_scores = scores.masked_fill(
            ~same_location,
            float("-inf"),
        )
        local_indices = torch.topk(
            local_scores,
            k=DENSE_LOCAL_DEPTH,
            dim=1,
            largest=True,
            sorted=True,
        ).indices.cpu().numpy()

        for row_number in range(end - start):
            query_location = validation_locations[
                start + row_number
            ]
            dense_global_rankings.append(
                global_indices[row_number].tolist()
            )
            dense_local_rankings.append(
                [
                    index
                    for index in local_indices[row_number].tolist()
                    if item_locations[index] == query_location
                ]
            )
        del scores, local_scores, same_location

print("Fine-tuned dense rankings готовы:", len(dense_global_rankings))

# после поиска освобождаем память перед построением bm25
del query_embeddings
del corpus_tensor
del query_tensor
del item_locations_tensor
del item_embeddings
del model
gc.collect()
torch.cuda.empty_cache()
print("Dense-модель и матрицы удалены из RAM")

In [ ]:
# строим bm25 по текстам объявлений и очищенным фильтрам
import bm25s
import Stemmer
from bm25s.tokenization import Tokenizer

# заголовок повторяем три раза, чтобы он сильнее влиял на поиск
titles = normalize_text(items["item_title_raw"])
params = normalize_text(
    items["item_infm_params_text"]
).str.slice(0, 700)
descriptions = normalize_text(
    items["item_description_raw"]
).str.slice(0, 800)
bm25_corpus_texts = (
    titles
    + " " + titles
    + " " + titles
    + " " + params
    + " " + descriptions
).tolist()

# эти названия полей не несут смысла и только засоряют запрос
filter_field_labels = [
    "срочная услуга (мультистатус)",
    "предмет или специальность",
    "кто оказывает услуги",
    "чем вы занимаетесь",
    "участие в пилоте cpl",
    "поиск по слотам",
    "рабочие дни",
    "ваши клиенты",
    "преподаватель",
    "занятия",
    "тип услуги",
    "вид услуги",
]
clean_filter_values = validation_queries[
    "normalized_params"
].copy()
for field_label in sorted(
    filter_field_labels,
    key=len,
    reverse=True,
):
    clean_filter_values = clean_filter_values.str.replace(
        field_label,
        " ",
        regex=False,
    )
clean_filter_values = (
    clean_filter_values
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)
bm25_query_texts = [
    query if not filter_text else f"{query} {filter_text}"
    for query, filter_text in zip(
        validation_queries["normalized_query"],
        clean_filter_values,
        strict=True,
    )
]

# разбиваем текст на токены и приводим русские слова к общей основе
stemmer = Stemmer.Stemmer("russian")
tokenizer = Tokenizer(
    lower=True,
    stopwords="russian",
    stemmer=stemmer,
)
corpus_tokens = tokenizer.tokenize(
    bm25_corpus_texts,
    update_vocab=True,
    return_as="tuple",
    show_progress=True,
)
retriever = bm25s.BM25(
    k1=1.5,
    b=0.75,
    method="lucene",
)
retriever.index(corpus_tokens, show_progress=True)
query_tokens = tokenizer.tokenize(
    bm25_query_texts,
    update_vocab=False,
    return_as="tuple",
    allow_empty=False,
    show_progress=True,
)

# заранее собираем объявления каждой локации
location_to_indices = {}
for document_index, location_id in enumerate(item_locations):
    location_to_indices.setdefault(
        int(location_id),
        [],
    ).append(document_index)
location_to_indices = {
    location_id: np.asarray(indices, dtype=np.int64)
    for location_id, indices in location_to_indices.items()
}

def top_positive_indices(
    scores: np.ndarray,
    k: int,
    candidates: np.ndarray | None = None,
) -> list[int]:
    # оставляем только документы с положительной оценкой bm25
    if candidates is None:
        eligible = np.flatnonzero(scores > 0)
        eligible_scores = scores[eligible]
    else:
        candidate_scores = scores[candidates]
        positive_mask = candidate_scores > 0
        eligible = candidates[positive_mask]
        eligible_scores = candidate_scores[positive_mask]

    if len(eligible) == 0:
        return []
    if len(eligible) > k:
        selected = np.argpartition(
            eligible_scores,
            -k,
        )[-k:]
        selected = selected[
            np.argsort(eligible_scores[selected])[::-1]
        ]
    else:
        selected = np.argsort(eligible_scores)[::-1]
    return eligible[selected].astype(np.int64).tolist()

# для каждого запроса сохраняем глобальную и локальную выдачу
bm25_global_rankings = []
bm25_local_rankings = []
for row_number, query_token_ids in enumerate(
    tqdm(query_tokens.ids, desc="BM25 rankings")
):
    scores = retriever.get_scores_from_ids(query_token_ids)
    bm25_global_rankings.append(
        top_positive_indices(
            scores,
            BM25_GLOBAL_DEPTH,
        )
    )
    local_candidates = location_to_indices.get(
        int(validation_locations[row_number]),
        np.empty(0, dtype=np.int64),
    )
    bm25_local_rankings.append(
        top_positive_indices(
            scores,
            BM25_LOCAL_DEPTH,
            local_candidates,
        )
    )

print("BM25 rankings готовы:", len(bm25_global_rankings))

# удаляем индекс и тексты, потому что дальше нужны только готовые списки
del titles
del params
del descriptions
del bm25_corpus_texts
del corpus_tokens
del query_tokens
del items
gc.collect()
print("Тексты и токены BM25 удалены из RAM")

In [ ]:
# проверяем разные квоты dense и bm25
# порядок чисел такой: local dense, local bm25, global bm25, global dense
# начинаем с основных конфигураций, которые хотим сравнить
configurations = {
    (32, 16, 2, 0),
    (32, 14, 4, 0),
    (48, 0, 0, 2),
    (50, 0, 0, 0),
    (0, 50, 0, 0),
}
for dense_local_quota in [
    20, 24, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46
]:
    for bm25_local_quota in [
        2, 4, 6, 8, 10, 12, 14, 16, 18, 20
    ]:
        for bm25_global_quota in [0, 2, 4, 5, 6, 8]:
            dense_global_quota = (
                50
                - dense_local_quota
                - bm25_local_quota
                - bm25_global_quota
            )
            if dense_global_quota >= 0:
                configurations.add(
                    (
                        dense_local_quota,
                        bm25_local_quota,
                        bm25_global_quota,
                        dense_global_quota,
                    )
                )

# добавляем кандидатов по очереди и не допускаем повторов
def add_unique_with_quota(
    result: list[int],
    candidates: list[int],
    quota: int,
) -> None:
    if quota <= 0:
        return
    added = 0
    seen = set(result)
    for document_index in candidates:
        if document_index not in seen:
            result.append(document_index)
            seen.add(document_index)
            added += 1
        if added == quota:
            break

# считаем recall и hit rate для каждой конфигурации
metrics_rows = []
for (
    dense_local_quota,
    bm25_local_quota,
    bm25_global_quota,
    dense_global_quota,
) in sorted(configurations):
    recalls = []
    hits = []

    for row_number, query_id in enumerate(validation_query_ids):
        predicted = []
        add_unique_with_quota(
            predicted,
            dense_local_rankings[row_number],
            dense_local_quota,
        )
        add_unique_with_quota(
            predicted,
            bm25_local_rankings[row_number],
            bm25_local_quota,
        )
        add_unique_with_quota(
            predicted,
            bm25_global_rankings[row_number],
            bm25_global_quota,
        )
        add_unique_with_quota(
            predicted,
            dense_global_rankings[row_number],
            dense_global_quota,
        )
        if len(predicted) < 50:
            add_unique_with_quota(
                predicted,
                dense_global_rankings[row_number],
                50 - len(predicted),
            )
        if len(predicted) != 50 or len(predicted) != len(set(predicted)):
            raise ValueError(
                "Не удалось сформировать 50 уникальных кандидатов"
            )

        relevant = relevant_indices[query_id]
        overlap = len(set(predicted) & relevant)
        recalls.append(overlap / len(relevant))
        hits.append(float(overlap > 0))

    metrics_rows.append(
        {
            "dense_local_quota": dense_local_quota,
            "bm25_local_quota": bm25_local_quota,
            "bm25_global_quota": bm25_global_quota,
            "dense_global_quota": dense_global_quota,
            "recall_at_50": float(np.mean(recalls)),
            "hit_rate_at_50": float(np.mean(hits)),
            "validation_queries": len(validation_queries),
        }
    )

# сортируем варианты по recall и сохраняем всю таблицу
metrics = pd.DataFrame(metrics_rows).sort_values(
    "recall_at_50",
    ascending=False,
    kind="stable",
).reset_index(drop=True)
metrics.to_csv(METRICS_PATH, index=False)
display(metrics.head(30))
print("Проверено конфигураций:", len(metrics))
print("Метрики сохранены:", METRICS_PATH)

In [ ]:
# показываем лучший вариант и конфигурацию финального решения
best = metrics.iloc[0]
final_configuration = metrics[
    (metrics["dense_local_quota"] == 32)
    & (metrics["bm25_local_quota"] == 14)
    & (metrics["bm25_global_quota"] == 4)
    & (metrics["dense_global_quota"] == 0)
].iloc[0]

print(
    "Лучший validation Recall@50:",
    round(float(best["recall_at_50"]), 6),
)
print("Лучшие настройки:")
print(best.to_string())
print()
print("Настройки финального решения 32/14/4/0:")
print(final_configuration.to_string())
